# Retrieval on the real knowledge bases

Compares BM25, dense embeddings, hybrid fusion and a cross-encoder reranker on TechQA (IBM Technotes) and Stack Exchange. Use a **T4 GPU** runtime (Runtime > Change runtime type).

In [ ]:
REPO_URL = 'https://github.com/rushitpatel100-oss/ticket-triage-ai.git'
import os
if not os.path.exists('/content/ticket-triage-ai'):
    !git clone -q $REPO_URL /content/ticket-triage-ai
%cd /content/ticket-triage-ai
!git pull -q
!git log --oneline -1
!pip install -q 'sentence-transformers>=3.0' 'nltk>=3.8'
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# Real data (skips the download if the files are already here)
import os
flag = '--skip-download' if os.path.exists('data/raw/techqa/corpus.zip') else ''
!python -m src.sources --only techqa stackexchange $flag

In [ ]:
# Two rerankers: MiniLM (small, trained on web search) and BGE base (larger).
# TechQA: all 910 questions. Stack Exchange: 2,000 questions per split to keep reranking short.
RERANKERS = 'cross-encoder/ms-marco-MiniLM-L-6-v2 BAAI/bge-reranker-base'
!python -m src.retrieval --datasets techqa --rerankers $RERANKERS --fp16
!python -m src.retrieval --datasets stackexchange --rerankers $RERANKERS --fp16 --max-queries-per-split 2000

In [ ]:
import json
r = json.load(open('results/retrieval_metrics.json'))
print('METRICS_START'); print(json.dumps(r)); print('METRICS_END')

In [ ]:
from IPython.display import Image
Image('results/retrieval_comparison.png')